In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "data" / "balanced_experimental_dataset.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_FILE.resolve()}")

df = pd.read_csv(DATA_FILE, encoding="cp1252")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nDifficulty distribution:")
print(df["Difficulty"].value_counts())

display(df.head())

Dataset loaded successfully!
Shape: (111, 5)

Difficulty distribution:
Difficulty
Hard      37
Easy      37
Medium    37
Name: count, dtype: int64


,Question Number,Question,Answer,Category,Difficulty
0,87,Explain the principles of least privilege and ...,Least privilege limits access rights; defense ...,Security,Hard
1,202,What is a variable in programming?,A variable is a named storage location used to...,Programming Fundamentals,Easy
2,229,What is the software development life cycle?,The software development life cycle is a struc...,Software Engineering,Easy
3,168,Design a system for efficient storage and retr...,"Optimize for write-heavy loads, use time-based...",Database Systems,Hard
4,105,What is the difference between SQL and NoSQL d...,"SQL databases are structured, use SQL, and are...",Back-end,Medium


In [2]:
required_columns = ["Question", "Answer", "Category", "Difficulty"]

# Check that all required columns exist
missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Check missing values
print("Missing values:")
display(df.isnull().sum())

# Remove rows with missing essential values
knn_df = df.dropna(subset=required_columns).copy()

# Clean text columns
for col in required_columns:
    knn_df[col] = knn_df[col].astype(str).str.strip()

# Remove empty questions and answers
knn_df = knn_df[
    knn_df["Question"].ne("")
    & knn_df["Answer"].ne("")
    & knn_df["Category"].ne("")
    & knn_df["Difficulty"].ne("")
].copy()

# Remove duplicate questions
knn_df["_normalized_question"] = (
    knn_df["Question"]
    .str.casefold()
    .str.split()
    .str.join(" ")
)

knn_df = knn_df.drop_duplicates(
    subset="_normalized_question"
).reset_index(drop=True)

print("Clean dataset shape:", knn_df.shape)
print("\nDifficulty distribution after cleaning:")
display(knn_df["Difficulty"].value_counts())

display(knn_df.head())

Missing values:


Question Number    0
Question           0
Answer             0
Category           0
Difficulty         0
dtype: int64

Clean dataset shape: (109, 6)

Difficulty distribution after cleaning:


Difficulty
Hard      37
Easy      37
Medium    35
Name: count, dtype: int64

,Question Number,Question,Answer,Category,Difficulty,_normalized_question
0,87,Explain the principles of least privilege and ...,Least privilege limits access rights; defense ...,Security,Hard,explain the principles of least privilege and ...
1,202,What is a variable in programming?,A variable is a named storage location used to...,Programming Fundamentals,Easy,what is a variable in programming?
2,229,What is the software development life cycle?,The software development life cycle is a struc...,Software Engineering,Easy,what is the software development life cycle?
3,168,Design a system for efficient storage and retr...,"Optimize for write-heavy loads, use time-based...",Database Systems,Hard,design a system for efficient storage and retr...
4,105,What is the difference between SQL and NoSQL d...,"SQL databases are structured, use SQL, and are...",Back-end,Medium,what is the difference between sql and nosql d...


In [3]:
%pip install scikit-learn joblib

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 1.4 MB/s eta 0:00:06
   ------- -------------------------------- 1.6/8.4 MB 2.9 MB/s eta 0:00:03
   ----------------- ---------------------- 3.7/8.4 MB 5.1 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.4 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 8.0 MB/s  0:00:01
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   - -------------------------------------- 1.8/37.4 MB 36.5 MB/s eta 0:00:01
   ----- ---------------------------------- 5.5/37.4 MB 12.4 MB/s eta 0:00:03
   ------ --------------------------------- 6.3/37.4 MB 13.8 MB/s eta 0:00:03
   ------ ------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Combine each question with its category
knn_df["SearchText"] = (
    knn_df["Question"] + " " + knn_df["Category"]
)

# Convert text into numerical vectors
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(knn_df["SearchText"])

print("TF-IDF conversion completed.")
print("Number of questions:", X.shape[0])
print("Number of features:", X.shape[1])

TF-IDF conversion completed.
Number of questions: 109
Number of features: 704


In [ ]:
# Build the KNN model
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=min(5, len(knn_df))
)

knn_model.fit(X)

print("KNN model fitted successfully!")
print("Questions in model:", len(knn_df))

KNN model fitted successfully!
Questions in model: 109


In [ ]:
#Recommendation function

def recommend_similar_questions(
    question,
    top_n=3,
    difficulty=None,
    category=None
):
    if not str(question).strip():
        raise ValueError("Enter a question or topic.")

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    query_vector = vectorizer.transform([str(question).strip()])

    # Search all questions so optional filters can be applied afterwards
    distances, indices = knn_model.kneighbors(
        query_vector,
        n_neighbors=len(knn_df)
    )

    results = knn_df.iloc[indices[0]].copy()
    results["Similarity"] = 1 - distances[0]

    # Exclude exact matches to the search query
    normalized_query = " ".join(str(question).casefold().split())
    normalized_results = (
        results["Question"]
        .str.casefold()
        .str.split()
        .str.join(" ")
    )

    results = results[normalized_results != normalized_query]

    # Optional difficulty filter
    if difficulty is not None:
        results = results[
            results["Difficulty"].str.casefold()
            == difficulty.casefold()
        ]

    # Optional category filter
    if category is not None:
        results = results[
            results["Category"].str.casefold()
            == category.casefold()
        ]

    results = results.sort_values(
        "Similarity",
        ascending=False
    ).head(top_n)

    return results[
        ["Question", "Answer", "Category", "Difficulty", "Similarity"]
    ].reset_index(drop=True)

In [7]:
recommendations = recommend_similar_questions(
    question="What is object-oriented programming?",
    top_n=3
)

display(recommendations)

,Question,Answer,Category,Difficulty,Similarity
0,What is inheritance in object-oriented program...,Inheritance allows a class to reuse or extend ...,Object-Oriented Programming,Easy,0.864487
1,What is a class in object-oriented programming?,A class is a blueprint that defines the proper...,Object-Oriented Programming,Easy,0.864487
2,What is an object in programming?,An object is an instance of a class that can h...,Object-Oriented Programming,Easy,0.825017


In [8]:
recommendations = recommend_similar_questions(
    question="Explain database normalization.",
    top_n=5
)

display(recommendations)

,Question,Answer,Category,Difficulty,Similarity
0,What is normalization in database design?,Normalization organizes data into separate tab...,Database and SQL,Hard,0.497117
1,What is a database?,A database is an organized collection of data ...,Databases,Easy,0.234154
2,"What is database denormalization, and when is ...",Denormalization introduces redundancy for perf...,Database and SQL,Medium,0.199273
3,Explain the difference between INNER JOIN and ...,INNER JOIN returns matching rows from both tab...,Database and SQL,Medium,0.157715
4,Optimize database queries for a high-traffic w...,"Focus on indexing, query optimization, using c...",Database Systems,Hard,0.151829


In [ ]:
#Display available difficulty levels

print("Available difficulty levels:")
print(knn_df["Difficulty"].unique())

Available difficulty levels:
<StringArray>
['Hard', 'Easy', 'Medium']
Length: 3, dtype: str


In [12]:
display(
    recommend_similar_questions(
        question="What is a database?",
        top_n=3,
        difficulty="Easy"
    )
)

,Question,Answer,Category,Difficulty,Similarity
0,What is a primary key in a database?,A primary key is a column or combination of co...,Databases,Easy,0.275445
1,How do you optimize a SQL query for better per...,"Optimize using indexes, efficient SQL, limitin...",Database and SQL,Easy,0.198746
2,What is the difference between a compiler and ...,A compiler translates a program into another f...,Programming Fundamentals,Easy,0.000000


In [13]:
# Score answers based on keyword coverage and similarity to expected answers
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Build a separate TF-IDF model for expected answers
answer_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

expected_answers = knn_df["Answer"].astype(str).tolist()
answer_vectors = answer_vectorizer.fit_transform(expected_answers)

def extract_keywords(text):
    """Extract meaningful words for basic keyword coverage."""
    words = re.findall(r"\b[a-zA-Z][a-zA-Z0-9+#.-]*\b", str(text).lower())

    return {
        word for word in words
        if len(word) > 2 and word not in ENGLISH_STOP_WORDS
    }

print("Answer-scoring system prepared.")
print("Expected answers available:", len(expected_answers))

Answer-scoring system prepared.
Expected answers available: 109


In [14]:
# Score a user's answer against the expected answer for a given question

def score_answer(question, user_answer):
    """
    Compare a user's answer with the expected answer
    associated with a question in the dataset.
    """

    if not str(user_answer).strip():
        return {
            "score": 0,
            "similarity": 0,
            "keyword_coverage": 0,
            "matched_keywords": [],
            "missing_keywords": [],
            "feedback": "No answer provided. Review the expected answer and try again."
        }

    # Find the matching question in the dataset
    normalized_question = " ".join(str(question).casefold().split())

    matches = knn_df[
        knn_df["Question"].apply(
            lambda q: " ".join(str(q).casefold().split())
        ) == normalized_question
    ]

    if matches.empty:
        raise ValueError(
            "Question not found in the dataset. "
            "Use a question from knn_df['Question']."
        )

    expected_answer = matches.iloc[0]["Answer"]

    # Compare the user's answer and the expected answer
    answer_vectors_pair = answer_vectorizer.transform(
        [expected_answer, str(user_answer)]
    )

    similarity = float(
        cosine_similarity(
            answer_vectors_pair[0:1],
            answer_vectors_pair[1:2]
        )[0][0]
    )

    # Compare the important words in both answers
    expected_keywords = extract_keywords(expected_answer)
    user_keywords = extract_keywords(user_answer)

    if expected_keywords:
        matched_keywords = expected_keywords & user_keywords
        missing_keywords = expected_keywords - user_keywords
        coverage = len(matched_keywords) / len(expected_keywords)
    else:
        matched_keywords = set()
        missing_keywords = set()
        coverage = 0.0

    # Experimental scoring formula
    score = round(
        (0.60 * similarity + 0.40 * coverage) * 100
    )

    # Basic feedback
    if score >= 80:
        feedback = "Good answer. Review the missing concepts, if any, before moving on."
    elif score >= 50:
        feedback = "Partially correct. Review the expected answer and strengthen the missing concepts."
    else:
        feedback = "Your answer needs improvement. Study the expected answer and try again."

    return {
        "score": score,
        "similarity": round(similarity * 100, 2),
        "keyword_coverage": round(coverage * 100, 2),
        "matched_keywords": sorted(matched_keywords),
        "missing_keywords": sorted(missing_keywords),
        "expected_answer": expected_answer,
        "feedback": feedback
    }

In [16]:
# Example usage of the scoring function
test_question = knn_df.iloc[0]["Question"]
expected_answer = knn_df.iloc[0]["Answer"]

print("Question:")
print(test_question)

print("\nExpected answer:")
print(expected_answer)

user_answer = input("\nEnter a sample interview answer: ")

result = score_answer(test_question, user_answer)

print("\n--- SCORING RESULTS ---")
print("Score:", result["score"], "/ 100")
print("Text similarity:", result["similarity"], "%")
print("Keyword coverage:", result["keyword_coverage"], "%")
print("Matched keywords:", result["matched_keywords"])
print("Missing keywords:", result["missing_keywords"])
print("Feedback:", result["feedback"])

Question:
Explain the principles of least privilege and defense in depth.

Expected answer:
Least privilege limits access rights; defense in depth layers security. Both minimize attack surfaces and provide redundancy.

--- SCORING RESULTS ---
Score: 30 / 100
Text similarity: 33.98 %
Keyword coverage: 23.08 %
Matched keywords: ['defense', 'depth', 'privilege']
Missing keywords: ['access', 'attack', 'layers', 'limits', 'minimize', 'provide', 'redundancy', 'rights', 'security', 'surfaces']
Feedback: Your answer needs improvement. Study the expected answer and try again.


In [17]:
# Adjust difficulty using the average of up to the last 3 scores
DIFFICULTY_LEVELS = ["Easy", "Medium", "Hard"]

def choose_next_difficulty(current_difficulty, recent_scores):
    """
    Adjust difficulty using the average of up to the last 3 scores.
    """

    if current_difficulty not in DIFFICULTY_LEVELS:
        raise ValueError(
            f"Difficulty must be one of {DIFFICULTY_LEVELS}"
        )

    if not recent_scores:
        return current_difficulty

    recent_average = sum(recent_scores[-3:]) / len(recent_scores[-3:])
    current_index = DIFFICULTY_LEVELS.index(current_difficulty)

    if recent_average < 50:
        next_index = max(0, current_index - 1)
    elif recent_average >= 80:
        next_index = min(len(DIFFICULTY_LEVELS) - 1, current_index + 1)
    else:
        next_index = current_index

    return DIFFICULTY_LEVELS[next_index]

In [18]:
# Example usage of the difficulty adjustment function
examples = [
    ("Easy", [35]),
    ("Medium", [65, 70, 68]),
    ("Medium", [85, 88, 82]),
    ("Hard", [92, 90, 95])
]

for current_level, scores in examples:
    next_level = choose_next_difficulty(current_level, scores)

    print(
        f"Current: {current_level:6} | "
        f"Recent scores: {scores} | "
        f"Next difficulty: {next_level}"
    )

Current: Easy   | Recent scores: [35] | Next difficulty: Easy
Current: Medium | Recent scores: [65, 70, 68] | Next difficulty: Medium
Current: Medium | Recent scores: [85, 88, 82] | Next difficulty: Hard
Current: Hard   | Recent scores: [92, 90, 95] | Next difficulty: Hard


In [19]:
# Select the next question based on topic, difficulty, and previously asked questions
def select_next_question(
    topic,
    target_difficulty,
    asked_questions=None,
    category=None
):
    if asked_questions is None:
        asked_questions = set()

    asked_normalized = {
        " ".join(str(q).casefold().split())
        for q in asked_questions
    }

    # First try the selected difficulty and category
    results = recommend_similar_questions(
        question=topic,
        top_n=len(knn_df),
        difficulty=target_difficulty,
        category=category
    )

    # If the category has no suitable results, try all categories
    if results.empty and category is not None:
        results = recommend_similar_questions(
            question=topic,
            top_n=len(knn_df),
            difficulty=target_difficulty
        )

    if results.empty:
        return None

    results = results[
        ~results["Question"].apply(
            lambda q: " ".join(str(q).casefold().split())
            in asked_normalized
        )
    ]

    if results.empty:
        return None

    return results.iloc[0].to_dict()

In [20]:
#Simulate a few rounds of question selection and scoring

# Start at Medium
current_difficulty = "Medium"
recent_scores = []
asked_questions = set()

# Use a question from your dataset as the starting topic
current_question = knn_df.iloc[0]["Question"]

for round_number, simulated_score in enumerate([85, 72, 42], start=1):
    asked_questions.add(current_question)

    recent_scores.append(simulated_score)

    next_difficulty = choose_next_difficulty(
        current_difficulty,
        recent_scores
    )

    print(f"\nRound {round_number}")
    print("Question:", current_question)
    print("Simulated answer score:", simulated_score, "/ 100")
    print("Next difficulty:", next_difficulty)

    next_question = select_next_question(
        topic=current_question,
        target_difficulty=next_difficulty,
        asked_questions=asked_questions
    )

    if next_question is None:
        print("No new question found at this difficulty.")
        break

    current_question = next_question["Question"]
    current_difficulty = next_difficulty

    print("Next question:", current_question)


Round 1
Question: Explain the principles of least privilege and defense in depth.
Simulated answer score: 85 / 100
Next difficulty: Hard
Next question: Explain the SOLID principles in software engineering.

Round 2
Question: Explain the SOLID principles in software engineering.
Simulated answer score: 72 / 100
Next difficulty: Hard
Next question: Explain the difference between black-box and white-box testing.

Round 3
Question: Explain the difference between black-box and white-box testing.
Simulated answer score: 42 / 100
Next difficulty: Hard
Next question: Describe the differences between manual and automated testing.
